In [1]:
import pandas as pd
import numpy as np
import random
import time
import json
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel 
from transformers import logging

In [2]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
behaviors = behaviors.sample(frac=1, random_state=42)
train_behaviors = behaviors[:int(len(behaviors) * 0.8)]
valid_behaviors = behaviors[int(len(behaviors) * 0.8):]

news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

embedding = {}
f = open("train/train_entity_embedding.vec")
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embedding[word] = coefs
f.close()

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [7]:
class RecommendationDataset(Dataset):
    def __init__(self, behaviors, news_dict, embedding, tokenizer):
        self.behaviors = behaviors
        self.news_dict = news_dict
        self.embedding = embedding
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.behaviors)

    def padding(self, item, size):
        item = torch.stack(item)[:size]
        if item.size(0) < size:
            padding_length = size - item.size(0)
            padding_item = torch.zeros((padding_length, *item.shape[1:]))
            item = torch.cat((item, padding_item), dim=0)
        return item
    
    def extract_entities(self, news):
        _, _, _, _, _, title_entities, abstract_entities = self.news_dict[news]
        
        title_entities = "[]" if isinstance(title_entities, float) else title_entities
        abstract_entities = "[]" if isinstance(abstract_entities, float) else abstract_entities
        
        vector = []
        entities = json.loads(title_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]
        entities = json.loads(abstract_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]

        if len(vector) == 0:
            vector = torch.zeros((30, 100))
        else:
            vector = self.padding(vector, 30)
                
        return vector

    def extract_text(self, news):
        category, subcategory, title, abstract, _, _, _ = self.news_dict[news]
        
        title = "" if isinstance(title, float) else title
        abstract = "" if isinstance(abstract, float) else abstract
        encoding = self.tokenizer(
            text = category + " " + subcategory,
            text_pair = title + " " + abstract,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = 200
        )
        return encoding
    
    def __getitem__(self, index):
        _, _, clicked_news, impressions = self.behaviors.iloc[index]
        
        history_vectors = []
        history_encodings = []
        clicked_news = clicked_news.split()
        for news in clicked_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            history_vectors.append(vector)
            history_encodings.append(encoding)
        history_vectors = self.padding(history_vectors, 20)
        history_ids = [encoding.input_ids[0, :] for encoding in history_encodings]
        history_ids = self.padding(history_ids, 20)
        history_type = [encoding.token_type_ids[0, :] for encoding in history_encodings]
        history_type = self.padding(history_type, 20)
        history_mask = [encoding.attention_mask[0, :] for encoding in history_encodings]
        history_mask = self.padding(history_mask, 20)

        recommen_vectors = []
        recommen_encodings = []
        impressions = impressions.split()
        labels = torch.tensor([int(impression.split('-')[1]) for impression in impressions])
        impression_news = [impression.split('-')[0] for impression in impressions]
        for news in impression_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            recommen_vectors.append(vector)
            recommen_encodings.append(encoding)
        recommen_vectors = torch.stack(recommen_vectors)
        recommen_ids = [encoding.input_ids for encoding in recommen_encodings]
        recommen_ids = torch.cat(recommen_ids)
        recommen_type = [encoding.token_type_ids for encoding in recommen_encodings]
        recommen_type = torch.cat(recommen_type)
        recommen_mask = [encoding.attention_mask for encoding in recommen_encodings]
        recommen_mask = torch.cat(recommen_mask)

        packed = {
            'history_vectors':  history_vectors,
            'history_ids':      history_ids,
            'history_type':     history_type,
            'history_mask':     history_mask,
            'recommen_vectors': recommen_vectors,
            'recommen_ids':     recommen_ids,
            'recommen_type':    recommen_type,
            'recommen_mask':    recommen_mask,
        }
        
        return packed, labels
    
train_dataset = RecommendationDataset(train_behaviors, news_dict, embedding, tokenizer)
valid_dataset = RecommendationDataset(valid_behaviors, news_dict, embedding, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

In [8]:
class RecommendationModel(nn.Module):
    def __init__(self):
        super(RecommendationModel, self).__init__()
        
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.linear = nn.Linear(200, 15)
    
    def forward(self, history_vectors, history_ids, history_type, history_mask, \
        recommen_vectors, recommen_ids, recommen_type, recommen_mask):
        return self.linear(history_ids[:, 0, :])
        

In [ ]:
device = 'cuda'
num_epoch = 5
show_freq = 10

model = RecommendationModel()
model = model.to(device)

loss = nn.MultiLabelSoftMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

best_valid_auc = 0.0
for epoch in range(num_epoch):
    epoch_start_time = time.time()
    train_loss, valid_loss = 0.0, 0.0
    train_count, valid_count = 0.0, 0.0
    train_true, valid_true = [], []
    train_pred, valid_pred = [], []

    model.train()
    for i, (packed, labels) in enumerate(train_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
            
        outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        model.zero_grad()
        
        train_loss += batch_loss.item()
        train_count += labels.shape[0] * labels.shape[1]
        train_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        train_pred += outputs.reshape(-1).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(train_loader):
            train_auc = roc_auc_score(train_true, train_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(train_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(train_auc, train_loss/train_count)
            )
            
    model.eval()
    for i, (packed, labels) in enumerate(valid_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        
        valid_loss += batch_loss.item()
        valid_count += labels.shape[0] * labels.shape[1]
        valid_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        valid_pred += outputs.reshape(-1).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(valid_loader):
            valid_auc = roc_auc_score(valid_true, valid_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(valid_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(valid_auc, valid_loss/valid_count)
            )
            
    if best_valid_auc < valid_auc:
        best_valid_auc = valid_auc
        torch.save(model.state_dict(), 'best_weight.pth')
        
print(f"Best Validation AUC: {best_valid_auc}")
        

[01/05 - 0010/14265] 5.05 sec Train AUC: 0.48 Loss: 3.2434 
[01/05 - 0020/14265] 10.56 sec Train AUC: 0.49 Loss: 3.2367 
[01/05 - 0030/14265] 15.81 sec Train AUC: 0.50 Loss: 3.1399 
[01/05 - 0040/14265] 20.71 sec Train AUC: 0.50 Loss: 3.0918 
[01/05 - 0050/14265] 25.12 sec Train AUC: 0.50 Loss: 3.1603 
[01/05 - 0060/14265] 30.70 sec Train AUC: 0.50 Loss: 3.1616 
[01/05 - 0070/14265] 35.43 sec Train AUC: 0.49 Loss: 3.1530 
[01/05 - 0080/14265] 41.26 sec Train AUC: 0.49 Loss: 3.1685 
[01/05 - 0090/14265] 46.18 sec Train AUC: 0.49 Loss: 3.1420 
[01/05 - 0100/14265] 51.08 sec Train AUC: 0.49 Loss: 3.1311 
[01/05 - 0110/14265] 56.08 sec Train AUC: 0.50 Loss: 3.1181 
[01/05 - 0120/14265] 61.74 sec Train AUC: 0.49 Loss: 3.1065 
[01/05 - 0130/14265] 66.89 sec Train AUC: 0.49 Loss: 3.0962 
[01/05 - 0140/14265] 71.73 sec Train AUC: 0.49 Loss: 3.0799 
[01/05 - 0150/14265] 76.91 sec Train AUC: 0.49 Loss: 3.0599 
[01/05 - 0160/14265] 82.04 sec Train AUC: 0.50 Loss: 3.0621 
[01/05 - 0170/14265] 86.5

[01/05 - 1340/14265] 693.40 sec Train AUC: 0.50 Loss: 2.1127 
[01/05 - 1350/14265] 698.91 sec Train AUC: 0.50 Loss: 2.1083 
[01/05 - 1360/14265] 703.36 sec Train AUC: 0.50 Loss: 2.1032 
[01/05 - 1370/14265] 708.56 sec Train AUC: 0.50 Loss: 2.0981 
[01/05 - 1380/14265] 714.12 sec Train AUC: 0.50 Loss: 2.0935 
[01/05 - 1390/14265] 718.82 sec Train AUC: 0.50 Loss: 2.0876 
[01/05 - 1400/14265] 723.69 sec Train AUC: 0.50 Loss: 2.0825 
[01/05 - 1410/14265] 728.49 sec Train AUC: 0.50 Loss: 2.0769 
[01/05 - 1420/14265] 733.14 sec Train AUC: 0.50 Loss: 2.0725 
[01/05 - 1430/14265] 738.33 sec Train AUC: 0.50 Loss: 2.0677 
[01/05 - 1440/14265] 743.86 sec Train AUC: 0.50 Loss: 2.0636 
[01/05 - 1450/14265] 749.16 sec Train AUC: 0.50 Loss: 2.0593 
[01/05 - 1460/14265] 754.15 sec Train AUC: 0.50 Loss: 2.0548 
[01/05 - 1470/14265] 759.15 sec Train AUC: 0.50 Loss: 2.0517 
[01/05 - 1480/14265] 764.13 sec Train AUC: 0.50 Loss: 2.0475 
[01/05 - 1490/14265] 769.69 sec Train AUC: 0.50 Loss: 2.0432 
[01/05 -

In [6]:
outputs.shape, labels.shape, packed['history_ids'].shape

(torch.Size([8, 15]), torch.Size([8, 15]), torch.Size([8, 20, 200]))